# 1 - Setup

In [ ]:
%%capture
%pip install --quiet pydantic-ai nest_asyncio
import nest_asyncio
nest_asyncio.apply() #Jupyter notebook-specific fix

In [ ]:
#TODO Set your OpenRouter API key
import os
os.environ['OPENROUTER_API_KEY'] = 'YOUR_KEY_GOES_HERE'

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL = OpenAIChatModel(
    'openai/gpt-4o-mini',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
)

# 2 - Extracting information without structured outputs


## 2.1

In [ ]:
from pydantic_ai import Agent

abstract = """
In this study, we introduce a novel deep learning approach for predicting protein-protein interactions (PPIs) in Saccharomyces cerevisiae and Mus musculus.
Our method leverages graph neural networks to capture complex molecular interactions and achieves an AUC-ROC score of 0.92 and AUC-PR score of 0.86 on the independent test set.
The model outperforms traditional machine learning methods and provides interpretable insights into key interacting residues.
Published in Journal of Computational Biology, 2023.
"""

plain_agent = Agent(
    MODEL,
    system_prompt="Extract the journal_name, published year, organisms, method, and metrics from the provided abstract.",
    output_type=str #this is the default
)
result = plain_agent.run_sync(user_prompt=f'Abstract: {abstract}')

print(result.output) # Hard to parse and to work with!

## 2.2

In [ ]:
lines = result.output.split('\n')
metric_line = next((line for line in lines if line.startswith('Metrics:')), None)

if metric_line:
    parsed_metric = metric_line.replace('Metrics:', '').strip()
    print(f"Parsed Metric from raw response: {parsed_metric}")
else:
    print("Metric not found in raw response.")

# 3 - Structured output with PydanticAI

## 3.1

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field

class PaperSummary(BaseModel):
    published_year: int
    journal_name: str
    organisms: list[str] = Field(description="List of latin names of used organisms")

agent = Agent(
    model=MODEL, 
    output_type=PaperSummary
)
result = agent.run_sync(f"Abstract to parse: {abstract}")
structured_output = result.output
print(type(structured_output))
print(structured_output.published_year) #Directly returns the correct data type

In [ ]:
print(structured_output.model_dump_json(indent=2))
# Output can still be hallucinated or output UNKNOWN instead of None

# Exercise 1

In [ ]:
#TODO Make an agent that will extract the ML method, and a dictionary of metrics from the abstract (e.g. {"accuracy": 0.75, "mcc": 0.80})
# HINT: you can type dictionary keys and values
# example_dict: dict[str,int] = {"a": 1, "b": 2, "c": 3}
abstract = """
In this study, we introduce a novel deep learning approach for predicting protein-protein interactions (PPIs) in Saccharomyces cerevisiae and Mus musculus.
Our method leverages graph neural networks to capture complex molecular interactions and achieves an AUC-ROC score of 0.92 and AUC-PR score of 0.86 on the independent test set.
The model outperforms traditional machine learning methods and provides interpretable insights into key interacting residues.
Published in Journal of Computational Biology, 2023.
"""


# 4 - Validators

## 4.1

Validators allows you to define custom validation logic for your structured outputs.

In [ ]:
abstract_idna = """
In this study, we propose iDNA-ABF, a multi-scale deep biological language learning model
that enables the interpretable prediction of DNA methylations based on genomic sequences only.
Benchmarking comparisons show that iDNA-ABF outperforms state-of-the-art methods for
different methylation predictions, achieving an accuracy of 92.6%. By integrating an interpretable analysis mechanism, the model
helps map important sequential determinants to downstream biological functions.
"""

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent

class MethodPerformance(BaseModel):
    method_name: str | None #we can use None as an alternative type 
    accuracy: float

agent = Agent(
    model=MODEL, 
    output_type=MethodPerformance
)
non_validated_result = agent.run_sync(user_prompt=f"Abstract to parse: {abstract_idna}")
non_validated_result.output

## 4.2

In [ ]:
my_paper = MethodPerformance(method_name="BioDNA", accuracy=0.950)

compared = max(my_paper.accuracy, non_validated_result.output.accuracy)
print(f"Best solution accuracy: {compared}") # Wrong! 0.950 > 0.926

## 4.3

Let's validate that the accuracy is between 0 and 1. **Always**.

In [ ]:
from pydantic_ai import Agent, ModelRetry

validated_agent = Agent(
    model=MODEL, 
    output_type=MethodPerformance, 
    retries=2
)

@validated_agent.output_validator
def validate_accuracy(output: MethodPerformance) -> MethodPerformance:
    if not (0 <= output.accuracy <= 1):
        raise ModelRetry("Accuracy must be between 0 and 1.") # Feeds the error back to the agent to retry
    return output

validated_result = validated_agent.run_sync(user_prompt=f"Abstract to parse: {abstract_idna}")
validated_result.output

## 4.4

In [ ]:
def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)
            print(f'    └─ {kind}: {snippet}')

In [ ]:
pretty_print_trace(validated_result)

# Exercise 2

In [ ]:
PAPERS = [
    {
        "title": "Agentomics: An Agentic System that Autonomously Develops "
                "State-of-the-art Solutions for Biomedical Machine Learning Tasks",
        "doi": "https://doi.org/10.64898/2026.01.27.702049",
        "abstract": (
            "Automated machine learning tools still require substantial human "
            "oversight when applied to biomedical data. We present Agentomics, an "
            "autonomous LLM-driven framework that implements multiple modeling "
            "strategies, inserts validation checkpoints, and produces "
            "deployment-ready models without human intervention. Across 20 datasets "
            "spanning protein engineering, drug discovery, and regulatory genomics, "
            "Agentomics outperforms competing agentic systems and reaches "
            "state-of-the-art results on 11 of 20 established benchmarks."
        ),
    },
    {
        "title": "A graph neural network for predicting protein–protein interactions "
                "in Saccharomyces cerevisiae",
        "abstract": (
            "We introduce a graph neural network for predicting protein–protein "
            "interactions from sequence and structural features in Saccharomyces "
            "cerevisiae. The model captures higher-order molecular context through "
            "message passing over an interaction graph and achieves an AUC-ROC of "
            "0.92 on an independent test set, outperforming classical baselines. "
            "An interpretability analysis highlights the residues that most "
            "influence each predicted interaction. DOI: 10.3390/molecules27186135"
        ),
    },
    {
        "title": "Masked-language pretraining improves DNA methylation prediction",
        "doi": None,
        "abstract": (
            "We propose a multi-scale biological language model that predicts DNA "
            "methylation from genomic sequence alone. Benchmarking against "
            "state-of-the-art methods across several methylation types, our approach "
            "achieves an accuracy of 92.6% while remaining interpretable, mapping "
            "salient sequence determinants to downstream biological functions."
        ),
    },
        {
        "title": "Masked-language pretraining improves DNA methylation prediction",
        "doi": "10.9999/does-not-exist",
        "abstract": (
            "We propose a multi-scale biological language model that predicts DNA "
            "methylation from genomic sequence alone. Benchmarking against "
            "state-of-the-art methods across several methylation types, our approach "
            "achieves an accuracy of 92.6% while remaining interpretable, mapping "
            "salient sequence determinants to downstream biological functions."
        ),
    },
  ]


In [ ]:
import httpx

def is_valid_doi(doi: str) -> bool:
    clean_doi = doi.strip().removeprefix("https://doi.org/")
    resp = httpx.get(f"https://api.crossref.org/works/{clean_doi}", timeout=10)
    return resp.status_code == 200

#TODO Given a list of PAPERS, extract the DOIs. The DOIs must be valid.